# Healthcare Test Results Prediction
## Data Analytics & Machine Learning Project

**Dataset**: Synthetic Healthcare Dataset (Kaggle)

**Problem Type**: Multiclass Classification

**Target Variable**: Test Results (Normal / Abnormal / Inconclusive)

**Objective**: Predict the test outcome of a hospital patient based on demographic, clinical, and administrative features.


## Section 1 — Import Libraries

In [ ]:
import os, warnings, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
)

warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries loaded successfully')

## Section 2 — Load Dataset

In [ ]:
df = pd.read_csv('../data/healthcare_dataset.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Columns:', df.columns.tolist())
print()
print(df.dtypes)

In [ ]:
df.describe(include='all')

## Section 3 — Data Quality Analysis

In [ ]:
print('Missing Values:')
print(df.isnull().sum())
print()
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Negative Billing Amounts: {(df["Billing Amount"] < 0).sum()}')

In [ ]:
df = df.drop_duplicates()
print(f'After removing duplicates: {df.shape[0]} rows')

## Section 4 — Feature Engineering

In [ ]:
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'])
df['Discharge Date']    = pd.to_datetime(df['Discharge Date'])
df['Length of Stay']    = (df['Discharge Date'] - df['Date of Admission']).dt.days
df['Admission Year']    = df['Date of Admission'].dt.year
df['Admission Month']   = df['Date of Admission'].dt.month
df['Admission DayOfWeek'] = df['Date of Admission'].dt.dayofweek
print('New features created: Length of Stay, Admission Year, Admission Month, Admission DayOfWeek')
df[['Length of Stay','Admission Year','Admission Month','Admission DayOfWeek']].describe()

## Section 5 — Exploratory Data Analysis

In [ ]:
print('Target Distribution:')
print(df['Test Results'].value_counts())
fig, ax = plt.subplots(figsize=(7,4))
vals = df['Test Results'].value_counts()
ax.bar(vals.index, vals.values, color=['#3b82d4','#e05c5c','#5cb85c'], edgecolor='black')
ax.set_title('Distribution of Test Results (Target Variable)')
ax.set_xlabel('Test Result'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].hist(df['Age'], bins=30, color='#3b82d4', edgecolor='black')
axes[0].set_title('Age Distribution'); axes[0].set_xlabel('Age')
axes[1].hist(df['Billing Amount'], bins=40, color='#5cb85c', edgecolor='black')
axes[1].set_title('Billing Amount Distribution'); axes[1].set_xlabel('$')
plt.tight_layout(); plt.show()

In [ ]:
ct = pd.crosstab(df['Medical Condition'], df['Test Results'])
ct.plot(kind='bar', figsize=(10,5), color=['#e05c5c','#f0ad4e','#5cb85c'])
plt.title('Test Results by Medical Condition')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
num_cols = ['Age','Billing Amount','Room Number','Length of Stay','Admission Year','Admission Month','Admission DayOfWeek']
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout(); plt.show()

## Section 6 — Data Preprocessing & Pipeline

In [ ]:
TARGET = 'Test Results'
DROP_COLS = ['Name','Doctor','Hospital','Date of Admission','Discharge Date']
X = df.drop(columns=DROP_COLS + [TARGET])
y = df[TARGET]

numerical_features   = ['Age','Billing Amount','Room Number','Length of Stay',
                         'Admission Year','Admission Month','Admission DayOfWeek']
categorical_features = ['Gender','Blood Type','Medical Condition','Insurance Provider',
                         'Admission Type','Medication']

numerical_pipeline   = Pipeline([('imputer', SimpleImputer(strategy='median')),
                                   ('scaler',  StandardScaler())])
categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                                   ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))])
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline,   numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])
print('Preprocessor pipeline ready')

## Section 7 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## Section 8 — Train Multiple Models

In [ ]:
models = {
    'Logistic Regression':  LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, random_state=42),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=100, random_state=42),
}

results = []
trained_pipelines = {}

for name, clf in models.items():
    print(f'Training: {name} ...')
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    results.append({'Model': name, 'Accuracy': round(acc,4), 'F1': round(f1,4)})
    trained_pipelines[name] = pipe
    print(f'  Accuracy={acc:.4f} | F1={f1:.4f}')

## Section 9 — Model Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values('F1', ascending=False)
print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8,4))
x = range(len(results_df))
ax.bar([i-0.2 for i in x], results_df['Accuracy'], 0.4, label='Accuracy', color='#3b82d4')
ax.bar([i+0.2 for i in x], results_df['F1'], 0.4, label='F1-Score', color='#e05c5c')
ax.set_xticks(list(x)); ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
ax.set_ylim(0,0.6); ax.set_title('Model Comparison'); ax.legend()
plt.tight_layout(); plt.show()

## Section 10 — Best Model & Evaluation

In [ ]:
best_name = results_df.iloc[0]['Model']
best_pipe = trained_pipelines[best_name]
y_pred_best = best_pipe.predict(X_test)
print(f'Best Model: {best_name}')
print(classification_report(y_test, y_pred_best))

cm = confusion_matrix(y_test, y_pred_best, labels=best_pipe.classes_)
fig, ax = plt.subplots(figsize=(7,5))
ConfusionMatrixDisplay(cm, display_labels=best_pipe.classes_).plot(ax=ax, cmap='Blues')
ax.set_title(f'Confusion Matrix - {best_name}')
plt.tight_layout(); plt.show()

## Section 11 — Feature Importance

In [ ]:
clf_step = best_pipe.named_steps['classifier']
if hasattr(clf_step, 'feature_importances_'):
    all_feat_names = numerical_features + categorical_features
    fi_df = pd.DataFrame({'Feature': all_feat_names, 'Importance': clf_step.feature_importances_})
    fi_df = fi_df.sort_values('Importance')
    fig, ax = plt.subplots(figsize=(8,6))
    ax.barh(fi_df['Feature'], fi_df['Importance'], color='#3b82d4')
    ax.set_title(f'Feature Importance - {best_name}')
    plt.tight_layout(); plt.show()
else:
    print('Feature importance not available for this model')

## Section 12 — Save & Test Model

In [ ]:
joblib.dump(best_pipe, '../model/trained_model.pkl')
print('Model saved: ../model/trained_model.pkl')

loaded = joblib.load('../model/trained_model.pkl')
sample = X_test.iloc[:3].copy()
preds  = loaded.predict(sample)
print('Sample predictions:', preds)
print('Actual labels:     ', y_test.iloc[:3].values)